# Notebook 2: Business Analysis

This notebook answers core business questions about the cleaned customer dataset using a mix of pandas and SQL queries.

**Questions we'll answer:**
1. Who are the biggest spenders, and what do they look like?
2. Do families with children spend differently than households without?
3. Which sales channel performs best, web, store, or catalog?
4. Which past campaigns actually worked, and on whom?
5. What's the spending profile by education level?

**Approach:** We'll load the cleaned dataset from Notebook 1, then use SQL (via pandas + SQLite) to answer each question.

In [1]:
import pandas as pd
import sqlite3

# Load the cleaned dataset we saved at the end of Notebook 1
df = pd.read_csv('marketing_campaign_clean.csv')

# Set up an in-memory SQLite database so we can run real SQL queries on the dataframe
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False, if_exists='replace')

# Confirm everything loaded
print("Dataset loaded:", df.shape)
print("SQL table created: 'customers'")
print("Columns available for querying:", len(df.columns))

Dataset loaded: (2240, 36)
SQL table created: 'customers'
Columns available for querying: 36


## Q1: Who Are the Biggest Spenders?

Identifying the customers in the top 10% by total spending and comparing them to the bottom 10%, looking at:
- Average income
- Average age
- Whether they have children
- How they prefer to shop (web vs catalog vs store)
- Their campaign response rate

**Why this matters:** If we can describe what big spenders look like, we can target similar profiles in future campaigns instead of casting a wide net.

In [2]:
# SQL query: compare top 10% vs bottom 10% of spenders across key dimensions
query = """
WITH spending_tiers AS (
    SELECT 
        *,
        NTILE(10) OVER (ORDER BY Total_Spent) AS spending_decile
    FROM customers
)
SELECT 
    CASE 
        WHEN spending_decile = 10 THEN 'Top 10% (Big Spenders)'
        WHEN spending_decile = 1  THEN 'Bottom 10% (Low Spenders)'
    END AS customer_tier,
    COUNT(*)                         AS num_customers,
    ROUND(AVG(Total_Spent), 0)       AS avg_total_spent,
    ROUND(AVG(Income), 0)            AS avg_income,
    ROUND(AVG(Age), 1)               AS avg_age,
    ROUND(AVG(Has_Children), 2)      AS pct_with_children,
    ROUND(AVG(NumWebPurchases), 1)   AS avg_web_purchases,
    ROUND(AVG(NumCatalogPurchases), 1) AS avg_catalog_purchases,
    ROUND(AVG(NumStorePurchases), 1) AS avg_store_purchases,
    ROUND(AVG(Total_Campaigns_Accepted), 2) AS avg_campaigns_accepted
FROM spending_tiers
WHERE spending_decile IN (1, 10)
GROUP BY customer_tier
ORDER BY avg_total_spent DESC;
"""

# Run the query against our SQLite database
result = pd.read_sql_query(query, conn)
result

,customer_tier,num_customers,avg_total_spent,avg_income,avg_age,pct_with_children,avg_web_purchases,avg_catalog_purchases,avg_store_purchases,avg_campaigns_accepted
0,Top 10% (Big Spenders),224,1827.0,79874.0,45.5,0.18,5.3,6.4,8.1,1.54
1,Bottom 10% (Low Spenders),224,22.0,30732.0,42.1,0.87,0.9,0.1,2.4,0.11


## Q2: Do Families Spend Differently Across Product Categories?

We know families with children spend less overall, but do they spend differently by category?
For example, do families with kids spend more on sweets or fish, but less on wine?

Comparing average spending in each of the 6 product categories between households with and without children.

In [3]:
# Compare average spending per category between households with vs without children
query = """
SELECT 
    CASE WHEN Has_Children = 1 THEN 'With Children' ELSE 'No Children' END AS household_type,
    COUNT(*)                       AS num_customers,
    ROUND(AVG(MntWines), 0)        AS avg_wine_spend,
    ROUND(AVG(MntMeatProducts), 0) AS avg_meat_spend,
    ROUND(AVG(MntFishProducts), 0) AS avg_fish_spend,
    ROUND(AVG(MntFruits), 0)       AS avg_fruit_spend,
    ROUND(AVG(MntSweetProducts),0) AS avg_sweets_spend,
    ROUND(AVG(MntGoldProds), 0)    AS avg_gold_spend,
    ROUND(AVG(Total_Spent), 0)     AS avg_total_spend
FROM customers
GROUP BY household_type
ORDER BY avg_total_spend DESC;
"""

result = pd.read_sql_query(query, conn)
result

,household_type,num_customers,avg_wine_spend,avg_meat_spend,avg_fish_spend,avg_fruit_spend,avg_sweets_spend,avg_gold_spend,avg_total_spend
0,No Children,638,487.0,373.0,77.0,52.0,53.0,64.0,1106.0
1,With Children,1602,231.0,85.0,22.0,16.0,17.0,36.0,407.0


## Q3: Channel Performance. Which Channel Drives the Most Value?

The retailer has four purchase channels: Web, Catalog, Store, and Deals.
We need to understand:
1. Which channel has the highest average **purchase volume** per customer?
2. Which channel correlates with **higher-spending** customers?
3. How does **website browsing behavior** (NumWebVisitsMonth) translate to actual purchases?

**Why this matters:** Marketing budget is finite. If catalog drives high-value customers but store drives volume, those require different campaign strategies and different investment levels.

In [4]:
# Compare channel performance: avg purchases per channel and how each channel correlates with total spend
query = """
SELECT 
    'Web'      AS channel, 
    ROUND(AVG(NumWebPurchases), 2)     AS avg_purchases_per_customer,
    ROUND(SUM(NumWebPurchases), 0)     AS total_purchases,
    ROUND(AVG(CASE WHEN NumWebPurchases     > 0 THEN Total_Spent END), 0) AS avg_spend_of_users,
    ROUND(100.0 * SUM(CASE WHEN NumWebPurchases     > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_customers_using
FROM customers
UNION ALL
SELECT 
    'Catalog', 
    ROUND(AVG(NumCatalogPurchases), 2),
    ROUND(SUM(NumCatalogPurchases), 0),
    ROUND(AVG(CASE WHEN NumCatalogPurchases > 0 THEN Total_Spent END), 0),
    ROUND(100.0 * SUM(CASE WHEN NumCatalogPurchases > 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM customers
UNION ALL
SELECT 
    'Store',
    ROUND(AVG(NumStorePurchases), 2),
    ROUND(SUM(NumStorePurchases), 0),
    ROUND(AVG(CASE WHEN NumStorePurchases   > 0 THEN Total_Spent END), 0),
    ROUND(100.0 * SUM(CASE WHEN NumStorePurchases   > 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM customers
UNION ALL
SELECT 
    'Deals',
    ROUND(AVG(NumDealsPurchases), 2),
    ROUND(SUM(NumDealsPurchases), 0),
    ROUND(AVG(CASE WHEN NumDealsPurchases   > 0 THEN Total_Spent END), 0),
    ROUND(100.0 * SUM(CASE WHEN NumDealsPurchases   > 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM customers
ORDER BY total_purchases DESC;
"""

result = pd.read_sql_query(query, conn)
result

,channel,avg_purchases_per_customer,total_purchases,avg_spend_of_users,pct_customers_using
0,Store,5.79,12970.0,607.0,99.3
1,Web,4.08,9150.0,615.0,97.8
2,Catalog,2.66,5963.0,800.0,73.8
3,Deals,2.33,5208.0,593.0,97.9


## Q4: Campaign Performance. What Worked, What Flopped, and Why?

The retailer ran 6 marketing campaigns. Each customer either responded (1) or ignored (0) each campaign.
We want to evaluate:

1. **Response rate by campaign** which campaigns hit and which missed?
2. **Customer responsiveness profile** are there "always-responders" (customers who accept multiple campaigns)?
3. **What do responders look like vs non-responders?** the demographic and behavioral profile that predicts engagement.

**Why this matters:** Knowing which past campaigns worked, on whom, lets us simulate the targeting strategy for the next campaign and project a lift in response rate.

In [5]:
# Q4a: Response rate for each of the 6 campaigns
query = """
SELECT 'Campaign 1' AS campaign, ROUND(100.0 * AVG(AcceptedCmp1), 2) AS response_rate_pct, SUM(AcceptedCmp1) AS responders FROM customers
UNION ALL
SELECT 'Campaign 2', ROUND(100.0 * AVG(AcceptedCmp2), 2), SUM(AcceptedCmp2) FROM customers
UNION ALL
SELECT 'Campaign 3', ROUND(100.0 * AVG(AcceptedCmp3), 2), SUM(AcceptedCmp3) FROM customers
UNION ALL
SELECT 'Campaign 4', ROUND(100.0 * AVG(AcceptedCmp4), 2), SUM(AcceptedCmp4) FROM customers
UNION ALL
SELECT 'Campaign 5', ROUND(100.0 * AVG(AcceptedCmp5), 2), SUM(AcceptedCmp5) FROM customers
UNION ALL
SELECT 'Campaign 6 (Latest)', ROUND(100.0 * AVG(Response), 2), SUM(Response) FROM customers
ORDER BY response_rate_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

,campaign,response_rate_pct,responders
0,Campaign 6 (Latest),14.91,334
1,Campaign 4,7.46,167
2,Campaign 3,7.28,163
3,Campaign 5,7.28,163
4,Campaign 1,6.43,144
5,Campaign 2,1.34,30


In [6]:
# Q4b: Profile of Campaign 6 responders vs non-responders
query = """
SELECT 
    CASE WHEN Response = 1 THEN 'Responded to Campaign 6' ELSE 'Did Not Respond' END AS responder_type,
    COUNT(*)                                  AS num_customers,
    ROUND(AVG(Income), 0)                     AS avg_income,
    ROUND(AVG(Age), 1)                        AS avg_age,
    ROUND(AVG(Has_Children), 2)               AS pct_with_children,
    ROUND(AVG(Total_Spent), 0)                AS avg_total_spent,
    ROUND(AVG(NumWebPurchases), 1)            AS avg_web_purchases,
    ROUND(AVG(NumCatalogPurchases), 1)        AS avg_catalog_purchases,
    ROUND(AVG(NumStorePurchases), 1)          AS avg_store_purchases,
    ROUND(AVG(Total_Campaigns_Accepted), 2)   AS avg_prior_campaigns_accepted,
    ROUND(AVG(Customer_Tenure_Days), 0)       AS avg_tenure_days
FROM customers
GROUP BY responder_type
ORDER BY avg_total_spent DESC;
"""

result = pd.read_sql_query(query, conn)
result

,responder_type,num_customers,avg_income,avg_age,pct_with_children,avg_total_spent,avg_web_purchases,avg_catalog_purchases,avg_store_purchases,avg_prior_campaigns_accepted,avg_tenure_days
0,Responded to Campaign 6,334,60182.0,44.6,0.49,987.0,5.1,4.2,6.1,1.99,447.0
1,Did Not Respond,1906,50851.0,45.3,0.75,539.0,3.9,2.4,5.7,0.18,337.0


## Q5: Spending Profile by Education Level

We already know that `Basic`-education customers earn roughly half the income of other groups.
But does education level predict *how* customers spend, beyond just how much?

For each education level, we'll look at:
- Average income
- Average total spending
- Spending efficiency (spend as % of income)
- Category preferences (wine vs meat vs fish, etc.)

**Why this matters:** If education predicts product preference (not just spending volume), the marketing team can use education as a targeting signal for premium product launches.

In [7]:
# Q5: Spending profile by education level
query = """
SELECT 
    Education,
    COUNT(*)                          AS num_customers,
    ROUND(AVG(Income), 0)             AS avg_income,
    ROUND(AVG(Total_Spent), 0)        AS avg_total_spent,
    ROUND(100.0 * AVG(Total_Spent) / AVG(Income), 2) AS spend_pct_of_income,
    ROUND(AVG(MntWines), 0)           AS avg_wine,
    ROUND(AVG(MntMeatProducts), 0)    AS avg_meat,
    ROUND(AVG(MntGoldProds), 0)       AS avg_gold,
    ROUND(100.0 * AVG(Response), 1)   AS campaign6_response_rate
FROM customers
GROUP BY Education
ORDER BY avg_total_spent DESC;
"""

result = pd.read_sql_query(query, conn)
result

,Education,num_customers,avg_income,avg_total_spent,spend_pct_of_income,avg_wine,avg_meat,avg_gold,campaign6_response_rate
0,PhD,486,56136.0,672.0,1.20,404.0,169.0,32.0,20.8
1,Graduation,1127,52714.0,620.0,1.18,284.0,179.0,51.0,13.5
2,Master,370,52891.0,612.0,1.16,333.0,163.0,40.0,15.4
3,2n Cycle,203,47621.0,497.0,1.04,198.0,141.0,46.0,10.8
4,Basic,54,20306.0,82.0,0.40,7.0,11.0,23.0,3.7
